<a href="https://colab.research.google.com/github/fakhreddine-jadib/App/blob/main/train_yolo_valves.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Valve Detection with YOLOv8

**Run this on a GPU runtime** (Colab: Runtime -> Change runtime type -> GPU). This will be extremely slow on CPU.

This trains on your real P&ID pages (tiled into 640x640 crops with remapped bounding boxes) instead of the
synthetic icon-classification approach from before. Two important notes about the source data:
- Only `valid` + `test` from your Roboflow export had label files (76 images total) — `train` had 185 images
  but zero labels, so it couldn't be used. This notebook's dataset is built from the 76 labeled images only,
  re-split 85/15, then tiled.
- Each large page (up to 6740x4767 px) was sliced into overlapping 640x640 tiles with boxes remapped, since
  your valve symbols are only ~25-100px on the full page — feeding a full page into YOLO would downscale
  valves into a few sub-pixels and the model would never learn to see them.
- A fraction of empty (no-valve) tiles were deliberately kept as hard negatives, so the model learns to
  *not* fire on pipes/text/instrument bubbles/blank space — this was the exact failure mode from your
  earlier CNN classifier approach.

In [1]:
!pip install -q ultralytics
!unzip -q yolo_tiled_dataset.zip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.6 MB/s eta 0:00:00


In [2]:
from ultralytics import YOLO

# start from a small COCO-pretrained checkpoint (transfer learning) rather than training from scratch
model = YOLO('yolov8n.pt')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
results = model.train(
    data='yolo_tiled/data.yaml',
    imgsz=640,
    epochs=100,
    batch=16,
    patience=20,       # early stop if val metrics stall
    project='valve_detector',
    name='run1',
)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_tiled/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie

## Check validation metrics

`mAP50` and `mAP50-95` are the key numbers. Given only 76 source pages, don't expect huge mAP right away —
the priority right now is confirming it (a) fires on real valve symbols and (b) mostly stays quiet on
non-valve regions. More labeled pages will matter more than tuning hyperparameters at this stage.

In [17]:
metrics = model.val()
print(metrics.box.map50, metrics.box.map)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 677.0±491.8 MB/s, size: 43.7 KB)
val: Scanning /content/yolo_tiled/val/labels.cache... 855 images, 586 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 855/855 143.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 4.9it/s 11.1s
                   all        855       1039      0.944      0.943      0.964      0.559
Speed: 2.4ms preprocess, 4.0ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /content/runs/detect/val-2
0.9639461984729583 0.5592123264457289


In [19]:
best_path = 'valve_detector/run1/weights/best.pt'
print(best_path)

valve_detector/run1/weights/best.pt


In [6]:
!pip install -q ultralytics

In [23]:
!find / -iname "best.pt" 2>/dev/null

/content/runs/detect/valve_detector/run1/weights/best.pt


In [24]:
from google.colab import files
files.download('runs/detect/valve_detector/run1/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('runs/detect/valve_detector/run1/weights/best.pt')   # path to the trained weights from the training notebook

TILE = 640
OVERLAP = 120
CONF_THRESHOLD = 0.35

img_path = 'pid_document.png'
full_img = cv2.imread(img_path)
full_rgb = cv2.cvtColor(full_img, cv2.COLOR_BGR2RGB)
H, W = full_rgb.shape[:2]
print(f"Page size: {W}x{H}")

Page size: 6740x4767


In [26]:
stride = TILE - OVERLAP
xs = list(range(0, max(W - TILE, 1), stride)) or [0]
ys = list(range(0, max(H - TILE, 1), stride)) or [0]
if xs[-1] + TILE < W:
    xs.append(max(W - TILE, 0))
if ys[-1] + TILE < H:
    ys.append(max(H - TILE, 0))

all_boxes = []
all_scores = []

for ty in ys:
    for tx in xs:
        tw = min(TILE, W - tx)
        th = min(TILE, H - ty)
        tile = full_rgb[ty:ty+th, tx:tx+tw]
        if tw < TILE or th < TILE:
            padded = np.full((TILE, TILE, 3), 255, dtype=np.uint8)
            padded[:th, :tw] = tile
            tile = padded

        results = model.predict(tile, conf=CONF_THRESHOLD, verbose=False)[0]
        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            all_boxes.append([tx + x1, ty + y1, tx + x2, ty + y2])
            all_scores.append(conf)

print(f"Raw detections across all tiles: {len(all_boxes)}")

Raw detections across all tiles: 236


In [27]:
# NMS to collapse duplicate detections where tiles overlapped
final_boxes = []
if all_boxes:
    rects = [[int(b[0]), int(b[1]), int(b[2]-b[0]), int(b[3]-b[1])] for b in all_boxes]
    indices = cv2.dnn.NMSBoxes(rects, all_scores, score_threshold=CONF_THRESHOLD, nms_threshold=0.3)
    for i in np.array(indices).flatten():
        final_boxes.append((all_boxes[i], all_scores[i]))

print(f"Final valve detections after NMS: {len(final_boxes)}")

Final valve detections after NMS: 142


In [28]:
result_img = full_rgb.copy()
for (x1, y1, x2, y2), conf in final_boxes:
    cv2.rectangle(result_img, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 3)
    cv2.putText(result_img, f"{conf*100:.0f}%", (int(x1), int(y1) - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

cv2.imwrite('pid_document_detected.png', cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
print("Saved pid_document_detected.png")

Saved pid_document_detected.png
